In [ ]:
!pip install datasets scikit-learn nltk

: 

In [ ]:
import re
import pandas as pd
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
ds = load_dataset("jquigl/imdb-genres")

In [ ]:
train_df = ds['train'].to_pandas()
validation_df = ds['validation'].to_pandas()

In [ ]:
train_df[['title', 'year']] = train_df['movie title - year'].str.rsplit(' - ', n=1, expand=True)
validation_df[['title', 'year']] = validation_df['movie title - year'].str.rsplit(' - ', n=1, expand=True)

In [ ]:
train_df = train_df.drop(['movie title - year' , 'genre' , 'year' , 'rating'] , axis= 1)
validation_df = validation_df.drop(['movie title - year' , 'genre' , 'year' , 'rating'] , axis= 1)

In [ ]:
train_df = train_df.rename(columns={'expanded-genres': 'genres',})
validation_df = validation_df.rename(columns={'expanded-genres': 'genres',})

In [ ]:
train_df

In [ ]:
stop_words = set(stopwords.words('english'))
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    filtered_words = []
    for w in words:
     if w not in stop_words:
        filtered_words.append(w)
    words = filtered_words
    return " ".join(words)

In [ ]:
train_df['description'] = train_df['description'].apply(clean_text)
train_df['genres'] = train_df['genres'].str.lower()
train_df['title'] = train_df['title'].str.lower()

In [ ]:
train_df = train_df.drop_duplicates(subset=['title', 'description', 'genres'])

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf.fit_transform(train_df['description'])

In [ ]:
indices = pd.Series(train_df.index, index=train_df['title'])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_movies(title=None, description=None, genre=None, top_n=5):
    results = None

    if title:
        title_clean = title.lower().strip()
        matches = train_df[train_df['title'].str.contains(title_clean, case=False, na=False)]
        if matches.empty:
            return "Movie title not found!"
        idx = matches.index[0]
        similarity_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix)[0]
        similarity_scores = [s for s in enumerate(similarity_scores) if s[0] != idx]
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
        movie_indices = [i[0] for i in similarity_scores]
        results = train_df.iloc[movie_indices]

    elif description:
        clean_description = clean_text(description)
        desc_vec = tfidf.transform([clean_description])
        similarity_scores = cosine_similarity(desc_vec, tfidf_matrix)[0]
        similarity_scores = list(enumerate(similarity_scores))
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
        movie_indices = [i[0] for i in similarity_scores]
        results = train_df.iloc[movie_indices]

    else:
        return "Please provide a movie title or description!"

    if genre:
        results = results[results['genres'].str.contains(genre.lower(), case=False, na=False)]

    return results[['title', 'description', 'genres']].head(top_n)

In [ ]:
recommend_movies(title = 'immigrant')

In [ ]:
recommend_movies(description = "3 friend in the college and their sucess story ")